# ۱. مقدمه

این پروژه یک سیستم تشخیص و ردیابی اشیاء ماژولار است که برای نظارت بر نوار نقاله طراحی شده است. هدف این معماری، تست الگوریتم‌ها روی سیستم‌های معمولی و در نهایت استقرار آسان آن‌ها بر روی دستگاه‌های Edge مانند ESP32-S3 است.

# ۲. معماری سیستم

معماری شامل اجزای زیر است:
- `CameraAdapter`: ارتباط با دوربین و دریافت فریم‌ها.
- `ModelAdapter`: اجرای مدل یادگیری ماشین (MediaPipe).
- `Detector`: هسته پردازشی در یک نخ (Thread) جداگانه.
- `Tracker`: ردیابی حرکت اشیاء در فریم‌ها.

# ۳. نصب و راه‌اندازی

برای اجرای این پروژه، ابتدا نیازمندی‌های ذکر شده در `requirements.txt` را نصب کنید.
```bash
pip install -r requirements.txt
```

In [ ]:
import sys
import os

# Add parent directory to path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

import cv2
import time
import matplotlib.pyplot as plt

%matplotlib inline

# ۴. تنظیمات دوربین

در اینجا عملکرد دوربین را با دریافت یک فریم تست می‌کنیم.

In [ ]:
from adapters.camera_adapter import USBCameraAdapter

camera = USBCameraAdapter(camera_index=0)
if camera.connect():
    print("Camera connected successfully!")
    frame = camera.capture_frame()
    if frame is not None:
        plt.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        plt.axis('off')
        plt.title("Camera Test Frame")
        plt.show()
    camera.release()
else:
    print("Failed to connect to camera.")

# ۵. مدل تشخیص

در این بخش، مدل TFLite را برای سیستم تشخیص دانلود و بارگذاری می‌کنیم.

In [ ]:
import urllib.request

model_url = "https://storage.googleapis.com/mediapipe-models/object_detector/efficientdet_lite0/int8/1/efficientdet_lite0.tflite"
model_dir = os.path.join(os.path.abspath(os.path.join(os.getcwd(), '..')), "models")
os.makedirs(model_dir, exist_ok=True)
model_path = os.path.join(model_dir, "efficientdet_lite0.tflite")

if not os.path.exists(model_path):
    print("Downloading model...")
    urllib.request.urlretrieve(model_url, model_path)
    print("Download complete!")
else:
    print("Model already exists.")

In [ ]:
from adapters.model_adapter import MediaPipeModelAdapter

model = MediaPipeModelAdapter(score_threshold=0.3)
model.load_model(model_path)
print("Model Info:", model.get_model_info())

# ۶. تشخیص اشیاء

یک فریم دریافت کرده و تشخیص را روی آن انجام می‌دهیم.

In [ ]:
from utils.visualizer import Visualizer

camera.connect()
frame = camera.capture_frame()
camera.release()

if frame is not None:
    detections = model.predict(frame)
    print(f"Detected {len(detections)} objects.")
    
    visualizer = Visualizer()
    result_frame = visualizer.draw_detections(frame, detections)
    
    plt.figure(figsize=(10, 8))
    plt.imshow(cv2.cvtColor(result_frame, cv2.COLOR_BGR2RGB))
    plt.axis('off')
    plt.show()

# ۷. ردیابی حرکت

ماژول ردیابی حرکت با استفاده از مرکز جرم (Centroid Tracker) را پیکربندی می‌کنیم.

In [ ]:
from core.tracker import SimpleCentroidTracker

tracker = SimpleCentroidTracker(max_distance=100.0, max_disappeared=5)
print("Tracker initialized.")

# ۸. تحلیل عملکرد

در اینجا کل پردازش (دوربین + مدل + ردیابی) به صورت پیوسته برای چند ثانیه اجرا شده و نرخ فریم بررسی می‌شود.

In [ ]:
from core.detector import ObjectDetector
from IPython.display import display, clear_output

camera = USBCameraAdapter(camera_index=0)
detector = ObjectDetector(camera=camera, model=model)
detector.start()

try:
    start_time = time.time()
    while time.time() - start_time < 5.0:
        frame, detections, fps = detector.get_latest_results()
        
        if frame is not None:
            tracks = tracker.update(detections)
            
            display_frame = frame.copy()
            display_frame = visualizer.draw_tracked_objects(display_frame, tracks)
            display_frame = visualizer.draw_fps(display_frame, fps)
            
            clear_output(wait=True)
            plt.figure(figsize=(6, 4))
            plt.imshow(cv2.cvtColor(display_frame, cv2.COLOR_BGR2RGB))
            plt.axis('off')
            plt.show()
            
        time.sleep(0.1)
finally:
    detector.stop()
    print("Test completed.")

# ۹. مراحل بعدی

جهت استقرار این پروژه روی ESP32-S3:
1. باید آداپتور دوربین برای استفاده از فریم‌ورک ESP-IDF بازنویسی شود.
2. از TensorFlow Lite for Microcontrollers جهت اجرای استنتاج بر روی سخت‌افزار لبه (Edge) استفاده گردد.
3. زبان توسعه به C++ تغییر می‌یابد اما منطق کلاس‌های طراحی شده ثابت می‌ماند.